# EPI v2.8.2 Investor Demo

This Google Colab notebook demonstrates the strongest current EPI capabilities:

- record a policy-backed AI workflow
- embed `policy.json` and `analysis.json` into a `.epi` artifact
- verify trust and integrity
- inspect the primary fault detected by the analyzer
- simulate tampering and show verification failure

EPI stands for **Evidence Packaged Infrastructure for AI**.

## 1. Install EPI

Use the PyPI path after release. Before release, switch `INSTALL_SOURCE` to `github` and point it at the repo branch you want to demo.

In [ ]:
INSTALL_SOURCE = "pypi"  # "pypi" or "github"
GITHUB_REF = "main"

if INSTALL_SOURCE == "pypi":
    %pip install -q epi-recorder==2.8.2
else:
    %pip install -q "git+https://github.com/mohdibrahimaiml/epi-recorder.git@{GITHUB_REF}"


## 2. Create a regulated policy and a workflow that violates it

This example uses a finance-style policy. The workflow will intentionally:

- exceed a financial limit
- skip a required action ordering
- exceed a threshold without human approval
- emit a prohibited secret-like token

That gives the fault analyzer something meaningful to detect.

In [ ]:
import json
from pathlib import Path

workspace = Path("/content/epi_colab_demo")
workspace.mkdir(parents=True, exist_ok=True)

policy = {
    "system_name": "loan-operations-agent",
    "system_version": "2.8.2-demo",
    "policy_version": "2026-03-17",
    "rules": [
        {
            "id": "R001",
            "name": "Do Not Exceed Balance",
            "severity": "critical",
            "description": "The agent must not approve an amount above the known balance.",
            "type": "constraint_guard",
            "watch_for": ["balance", "available_balance", "limit"],
            "violation_if": "approved_amount > balance"
        },
        {
            "id": "R002",
            "name": "Verify Identity Before Refund",
            "severity": "high",
            "description": "Identity verification must happen before refund.",
            "type": "sequence_guard",
            "required_before": "refund",
            "must_call": "verify_identity"
        },
        {
            "id": "R003",
            "name": "Human Approval Above 10000",
            "severity": "high",
            "description": "Amounts above 10,000 require human approval.",
            "type": "threshold_guard",
            "threshold_value": 10000,
            "threshold_field": "amount",
            "required_action": "human_approval"
        },
        {
            "id": "R004",
            "name": "Never Output Secrets",
            "severity": "critical",
            "description": "The agent must never emit secret-like values or API keys.",
            "type": "prohibition_guard",
            "prohibited_pattern": "sk-[A-Za-z0-9]+"
        }
    ]
}

(workspace / "epi_policy.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
print((workspace / "epi_policy.json").read_text(encoding="utf-8"))


In [ ]:
%%writefile /content/epi_colab_demo/investor_demo_workflow.py
from pathlib import Path
from epi_recorder import record

output = Path("/content/epi_colab_demo/colab_fault_demo.epi")

with record(
    str(output),
    workflow_name="Colab Investor Fault Demo",
    goal="Show that EPI records AI actions and pinpoints mistakes",
    notes="Intentional policy and heuristic violations for demo",
    metadata_tags=["colab", "investor-demo", "fault-analysis"],
) as epi:
    epi.log_step("tool.response", {
        "tool": "fetch_account_context",
        "account_id": "ACC-4401",
        "customer_id": "CUST-9001",
        "balance": 500.0,
        "currency": "USD",
    })

    epi.log_step("llm.error", {
        "error": "Rate limit exceeded while fetching sanctions data",
        "provider": "demo-llm",
    })

    epi.log_step("tool.call", {
        "tool": "approve_loan_disbursement",
        "amount": 2000.0,
        "account_id": "ACC-4401",
        "reason": "urgent disbursement",
    })

    epi.log_step("tool.call", {
        "tool": "process_refund",
        "amount": 200.0,
        "reference": "REF-2026-001",
    })

    epi.log_step("tool.call", {
        "tool": "approve_wire_transfer",
        "amount": 15000.0,
        "destination": "vendor-settlement",
    })

    epi.log_step("llm.response", {
        "text": "Escalation note: temporary key observed as sk-ABC123SECRET for retry flow.",
        "model": "demo-llm",
    })

    epi.log_step("tool.response", {
        "tool": "create_case",
        "case_reference": "CASE-7702",
        "status": "pending_manual_review",
    })

print(f"Created demo artifact: {output}")


In [ ]:
%cd /content/epi_colab_demo
!python investor_demo_workflow.py


## 3. Verify the artifact and inspect its contents

In [ ]:
!python -m epi_cli.main verify /content/epi_colab_demo/colab_fault_demo.epi
!python -m epi_cli.main analyze /content/epi_colab_demo/colab_fault_demo.epi


In [ ]:
import json
import zipfile
from pathlib import Path

artifact = Path('/content/epi_colab_demo/colab_fault_demo.epi')
with zipfile.ZipFile(artifact, 'r') as zf:
    print('Artifact contents:')
    for name in zf.namelist():
        print(' -', name)

    policy_json = json.loads(zf.read('policy.json').decode('utf-8'))
    analysis_json = json.loads(zf.read('analysis.json').decode('utf-8'))

print('\nPolicy rules embedded:', len(policy_json['rules']))
print('Primary fault:')
print(json.dumps(analysis_json['primary_fault'], indent=2))
print('\nAnalysis summary:')
print(json.dumps(analysis_json['summary'], indent=2))


## 4. Simulate tampering

Here we modify a sealed file inside the artifact and show that verification fails. This is one of the clearest demonstrations of EPI's trust model.

In [ ]:
import json
import zipfile
from pathlib import Path

original = Path('/content/epi_colab_demo/colab_fault_demo.epi')
tampered = Path('/content/epi_colab_demo/colab_fault_demo_tampered.epi')

with zipfile.ZipFile(original, 'r') as src, zipfile.ZipFile(tampered, 'w', compression=zipfile.ZIP_DEFLATED) as dst:
    for item in src.infolist():
        data = src.read(item.filename)
        if item.filename == 'analysis.json':
            payload = json.loads(data.decode('utf-8'))
            payload['fault_detected'] = False
            data = json.dumps(payload, indent=2).encode('utf-8')
        dst.writestr(item, data)

print('Created tampered artifact:', tampered)


In [ ]:
!python -m epi_cli.main verify /content/epi_colab_demo/colab_fault_demo_tampered.epi


## 5. Download the demo artifacts

You can download both the valid and tampered `.epi` files from Colab for investor walkthroughs or offline inspection.

In [ ]:
from google.colab import files

files.download('/content/epi_colab_demo/colab_fault_demo.epi')
# files.download('/content/epi_colab_demo/colab_fault_demo_tampered.epi')


## Investor takeaway

This notebook shows the three most important EPI capabilities:

1. **Portable evidence**: AI execution is sealed into a `.epi` artifact.
2. **Fault intelligence**: EPI identifies the primary mistake and explains why it matters.
3. **Tamper evidence**: if a sealed file changes, verification fails.
